# Gravitational Wave ML — Exploratory Analysis

This notebook explores the LIGO strain data, visualises the preprocessing pipeline,
and builds intuition for what the CNN will learn to detect.

## Contents
1. Download and inspect a real GW event (GW150914)
2. Visualise raw strain vs. bandpass-filtered vs. whitened
3. Q-transform spectrograms — signal vs. noise
4. Dataset class distribution
5. Feature distributions (Random Forest features)

In [3]:
import sys
sys.path.append('..')  # add project root

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Load GW150914 via gwpy

In [4]:
from gwpy.timeseries import TimeSeries

# GW150914 merger GPS time
GPS_MERGE = 1126259462.4

# Fetch 8 seconds around the merger from Hanford (H1)
strain_h1 = TimeSeries.fetch_open_data('H1', GPS_MERGE - 4, GPS_MERGE + 4,
                                        sample_rate=4096, verbose=True)
strain_l1 = TimeSeries.fetch_open_data('L1', GPS_MERGE - 4, GPS_MERGE + 4,
                                        sample_rate=4096, verbose=True)
print(f'H1 shape: {strain_h1.shape}  |  Sample rate: {strain_h1.sample_rate}')

ModuleNotFoundError: No module named 'gwpy'

## 2. Raw vs. Filtered vs. Whitened

In [ ]:
from src.data.preprocessing import bandpass_filter, whiten

raw   = np.array(strain_h1)
fs    = float(strain_h1.sample_rate.value)
t     = np.array(strain_h1.times) - GPS_MERGE

filt    = bandpass_filter(raw, fs, f_low=20, f_high=500)
white   = whiten(filt, fs)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(t, raw,   lw=0.4, color='steelblue')
axes[0].set_ylabel('Raw strain', fontsize=12)
axes[0].set_title('GW150914 — H1 strain', fontsize=13, fontweight='bold')

axes[1].plot(t, filt,  lw=0.5, color='darkorange')
axes[1].set_ylabel('Bandpass filtered\n(20–500 Hz)', fontsize=12)

axes[2].plot(t, white, lw=0.5, color='forestgreen')
axes[2].set_ylabel('Whitened', fontsize=12)
axes[2].set_xlabel('Time relative to merger (s)', fontsize=12)
axes[2].axvline(0, color='red', ls='--', lw=1.5, label='Merger time')
axes[2].legend(fontsize=11)

for ax in axes:
    ax.set_xlim([-2, 2])
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/gw150914_preprocessing.png', dpi=130)
plt.show()

## 3. Q-transform Spectrograms

In [ ]:
# Use gwpy's built-in Q-transform
qgram_h1 = strain_h1.q_transform(outseg=(GPS_MERGE-1, GPS_MERGE+0.5),
                                   frange=(20, 500), qrange=(4, 64))
qgram_l1 = strain_l1.q_transform(outseg=(GPS_MERGE-1, GPS_MERGE+0.5),
                                   frange=(20, 500), qrange=(4, 64))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, qgram, det in zip(axes, [qgram_h1, qgram_l1], ['H1 (Hanford)', 'L1 (Livingston)']):
    im = ax.imshow(
        np.array(qgram), origin='lower', aspect='auto',
        extent=[qgram.times.value[0] - GPS_MERGE,
                qgram.times.value[-1] - GPS_MERGE,
                qgram.frequencies.value[0],
                qgram.frequencies.value[-1]],
        cmap='viridis', vmin=0, vmax=25
    )
    ax.set_yscale('log')
    ax.set_xlabel('Time relative to merger (s)', fontsize=12)
    ax.set_ylabel('Frequency (Hz)', fontsize=12)
    ax.set_title(f'Q-transform — {det}', fontsize=13, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Normalised energy')

plt.tight_layout()
plt.savefig('../results/gw150914_qtransform.png', dpi=130)
plt.show()
print("Notice the characteristic 'chirp' — frequency rising rapidly toward merger")

## 4. Signal vs. Noise Spectrograms

In [ ]:
from src.data.preprocessing import compute_spectrogram, bandpass_filter, whiten
from src.data.synthetic import _load_psd, make_coloured_noise

rng = np.random.default_rng(42)
fs  = 4096.0
n   = int(fs)   # 1-second segment

psd = _load_psd(None, fs, n)
noise_seg = make_coloured_noise(n, fs, psd, rng)
noise_seg = whiten(bandpass_filter(noise_seg, fs), fs)

# Signal: use the whitened GW150914 around merger
merger_idx = int((GPS_MERGE - float(strain_h1.t0.value)) * fs)
sig_raw    = raw[max(0, merger_idx - n // 2) : merger_idx + n // 2]
sig_seg    = whiten(bandpass_filter(sig_raw, fs), fs)

spec_sig   = compute_spectrogram(sig_seg, fs)
spec_noise = compute_spectrogram(noise_seg, fs)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, spec, title, cmap in zip(
    axes,
    [spec_sig, spec_noise],
    ['BBH merger (GW150914)', 'Background noise'],
    ['viridis', 'plasma']
):
    ax.imshow(spec, origin='lower', aspect='auto', cmap=cmap)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Time bins')
    ax.set_ylabel('Frequency bins (log-scaled)')

plt.tight_layout()
plt.savefig('../results/signal_vs_noise_spectrograms.png', dpi=130)
plt.show()

## 5. Feature Distributions (Random Forest features)

In [ ]:
from src.models.random_forest_baseline import extract_features, BAND_EDGES_HZ

# Generate noise and synthetic BBH examples for comparison
n_examples = 200
noise_feats = np.stack([extract_features(make_coloured_noise(n, fs, psd, rng), fs)
                         for _ in range(n_examples)])

feature_names = [
    'RMS', 'Peak amp', 'Crest factor', 'Kurtosis', 'Skewness',
    'ZCR', 'Above 5σ', 'Peak freq', 'Peak PSD', 'SNR estimate',
    'Bandwidth', 'Mean freq', 'Freq sweep', 'Freq trend',
    'Band 20-60', 'Band 60-120', 'Band 120-240', 'Band 240-500'
]

# Highlight a few key discriminating features
key_features = [3, 6, 8, 9, 12, 13]   # kurtosis, above 5σ, SNR, peak PSD, sweep, trend

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()

for i, fi in enumerate(key_features):
    ax = axes[i]
    ax.hist(noise_feats[:, fi], bins=30, alpha=0.6, color='royalblue', label='Noise')
    ax.set_xlabel(feature_names[fi], fontsize=11)
    ax.set_ylabel('Count')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

fig.suptitle('Feature Distributions — Noise Background', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/feature_distributions.png', dpi=130)
plt.show()

## 6. Learning Curve (after training)

In [ ]:
import pandas as pd

log_path = '../logs/training_log.csv'
try:
    df = pd.read_csv(log_path)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0].plot(df['epoch'], df['val_loss'],   label='Val')
    axes[0].set_xlabel('Epoch');  axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss Curves');  axes[0].legend();  axes[0].grid(alpha=0.3)

    axes[1].plot(df['epoch'], df['train_acc'], label='Train')
    axes[1].plot(df['epoch'], df['val_acc'],   label='Val')
    axes[1].set_xlabel('Epoch');  axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy Curves');  axes[1].legend();  axes[1].grid(alpha=0.3)

    axes[2].plot(df['epoch'], df['val_f1'], color='purple', label='Val macro-F1')
    axes[2].set_xlabel('Epoch');  axes[2].set_ylabel('Macro F1')
    axes[2].set_title('Validation F1');  axes[2].legend();  axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('../results/learning_curves.png', dpi=130)
    plt.show()
except FileNotFoundError:
    print('No training log found yet — run: python main.py train --model cnn_spectrogram')